# Databricks notebook source
 
 # Transform Races Data

 1. Read bronze `races` table
 1. Keep only the columns required for analytics (Drop `url` column)
 1. Standardise column names using snake_case (`raceName` → `race_name`, `circuitId` → `circuit_id`)
 1. Rename columns to make them more meaningful (`date` → `race_date`)
 1. Remove duplicate records
 1. Transform values of column `race_name` to Title Case
 1. Write the transformed data to silver `races` table


- ## Step 1 - Read Bronze Table Data

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.table(bronze_table)

## Step 2 : Keep only columns required for Analytics(Drop URL)

In [0]:
races_selected_df = races_df.drop("url")

## Step 3 &4 - Standardize column name


In [0]:
races_renamed_df = (
    races_selected_df
    .withColumnsRenamed(
        {"raceName":"race_name",
         "circuitId":"circuit_id",
         "date":"race_date"
         }
    )
)

In [0]:
# display(races_renamed_df)

Databricks data profile. Run in Databricks to view.

## Step 5 - Filter out rows where primary key is NULL

In [0]:
races_valid_df = (
    races_renamed_df
    .filter(
        F.col("season").isNotNull() | F.col("round").isNotNull()

    )
)


In [0]:
# display(races_valid_df)

## Step 6 - Remove Duplicates

In [0]:
races_distinct_df = races_valid_df.dropDuplicates(["season","round"])

In [0]:
# display(races_distinct_df)

## Step 7 - Transform required column values to titlecase

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
    
)

In [0]:
# display(races_final_df)

## Step 8 - Write data to silver table

In [0]:
(
    races_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
# display(spark.table(silver_table))